# 情報リテラシー 講義(5) 演習ノートブック
## ネットワークの仕組みとインターネット (1)

- 講義日：2025-05-27（千葉大学Moodle 情報リテラシー(23)(24) T1-2・火4）
- 担当：檜垣 泰彦（アカデミック・リンク・センター 特任教授）
- 教科書：竹下隆史・村山公保・荒井透・苅田幸雄『マスタリングTCP/IP 入門編 第5版』オーム社（2012）

このノートブックは講義スライドに沿って手を動かして確認するための演習集です。各セルは Google Colab／JupyterLab でそのまま実行できます。

### 全体マップ
| 節 | テーマ | 対応するスライド章 |
|---|---|---|
| 1 | ネットワークの基本構成（LAN / WAN） | 第1部 概論 |
| 2 | プロトコルと OSI 参照モデル | 第1部 概論 |
| 3 | IPアドレスと二進数表現 | CH2 IPプロトコル |
| 4 | サブネットマスク | CH2 IPプロトコル |
| 5 | ルーターとデフォルトゲートウェイ | CH2 IPプロトコル |
| 6 | DNS（名前解決） | CH2 IPプロトコル |
| 7 | DHCP（設定の自動配布） | CH2 IPプロトコル |
| 8 | プライベートIPと NAT | CH2 IPプロトコル |
| 9 | ポート番号 | CH3 TCPとUDP |
| 10 | TCP / UDP（疑似通信） | CH3 TCPとUDP |
| 11 | まとめ＆発展（無線LANセキュリティ） | CH1 データリンク層 |

## 1. ネットワークの基本構成
ネットワークはその規模によって、**LAN**（Local Area Network）と **WAN**（Wide Area Network）に大別されます。

- **LAN：** 1つの建物や大学のキャンパスなど、限られた狭い地域でのネットワーク。
- **WAN：** 離れた地域のコンピュータや LAN 同士を接続した、広域のネットワーク。

私たちが普段利用するインターネットは、多数の組織や ISP（Internet Service Provider）のネットワークが IX（Internet Exchange）を介して相互接続されたものです。通信量が急増してネットワークの許容量を超え、つながりにくくなる現象を **輻輳（ふくそう）** と呼びます。

In [ ]:
# 【演習1】講義内容の確認クイズ（ネットワークの規模と通信現象）
print('--- ネットワーク基本クイズ ---')
answer1 = input('Q1. 大学のキャンパス内など限られた狭い地域のネットワークを何と呼びますか？ (アルファベット3文字で入力): ')
if answer1.strip().upper() == 'LAN':
    print('正解です！建物内などで個別に構築されたネットワークですね。')
else:
    print('不正解です。講義スライド2を復習しましょう。')

answer2 = input('Q2. 通信が集中し、ネットワークの許容量を超えてつながりにくくなる現象を何と言いますか？: ')
if answer2.strip() in ('輻輳', 'ふくそう'):
    print('正解です！道路の交通渋滞のような状態のことです。')
else:
    print('不正解です。ヒントは「ふくそう」です。')

## 2. プロトコルと OSI 参照モデル（7階層）
ネットワークを介して通信するために定められた約束事の集合を **プロトコル（通信規約）** と呼びます。メーカや OS が異なる機器同士でも、同じプロトコルを使えば互いに通信できます。

通信に必要な機能を整理するため、**OSI 参照モデル** では通信を 7 つの階層（レイヤ）に分けて考えます。

| 層 | 名称 | 代表的な装置 |
|---|---|---|
| 7 | アプリケーション層 | （ホスト/アプリ） |
| 6 | プレゼンテーション層 | 同上 |
| 5 | セッション層 | 同上 |
| 4 | トランスポート層 | 同上 |
| 3 | ネットワーク層 | **ルーター / L3スイッチ** |
| 2 | データリンク層 | **ブリッジ / L2スイッチ / ハブ** |
| 1 | 物理層 | **リピーター** |

送信側は上位層から順にデータへ「ヘッダ」を付加して **パケット** を作ります（**カプセル化**）。受信側は逆にヘッダを剥がして中身を取り出します。

In [ ]:
# 【演習2】文字列で体験する「階層型ヘッダ（カプセル化）」シミュレーション
print('--- カプセル化パズル ---')
data = input('送信したいメッセージ文字列を入力してください (例: Hello): ')

# 各階層を模したヘッダ情報を定義
t_header = '[TCP_Header:Port80]'
n_header = '[IP_Header:192.168.1.10]'
d_header = '[Ethernet_Header:MAC_Address]'

# 送信側：上位層から順にヘッダで包んでいく（カプセル化）
packet = d_header + n_header + t_header + '[' + data + ']'
print('\n[送信側] カプセル化されてネットワークを流れるパケットデータ:')
print(packet)

# 受信側：下位層から順にヘッダを剥いて中身を取り出す
print('\n[受信側] 各階層のヘッダを解析してデータを取り出します...')
extracted_data = packet.split('[')[-1].replace(']', '')
print(f'最終的にアプリケーションに届いたデータ: {extracted_data}')

## 3. IPアドレスの基本とアドレスの拡張（IPv4 / IPv6）
インターネットに接続されるすべてのホストには、識別子として **IPアドレス** が割り当てられます（グローバルIPは ICANN により世界で唯一の値が配分）。

- **IPv4：** 長さ **32 ビット**。インターネットの普及に伴い、アドレス枯渇問題が起きています。
- **IPv6：** 長さ **128 ビット** に大幅拡張。直接アクセス可能なアドレスの不足を抜本的に解消します。

コンピュータの内部では、これらはすべて `0` と `1` で構成される二進数（バイナリ）のビット列として処理されています。

In [ ]:
# 【演習3】IPアドレスをコンピュータが理解する「二進数」に変換してみよう
ip_input = input('十進数のIPv4アドレスを入力してください (例: 192.168.200.107): ')

try:
    octets = ip_input.split('.')
    if len(octets) != 4 or not all(0 <= int(o) <= 255 for o in octets):
        raise ValueError

    binary_octets = [format(int(octet), '08b') for octet in octets]

    print(f'\n入力された十進数表現: {ip_input}')
    print('各グループ（オクテット）を8ビットの二進数にした結果:')
    for i, bin_str in enumerate(binary_octets):
        print(f'  第{i+1}部分 ({octets[i]}): {bin_str}')

    combined_bin = '.'.join(binary_octets)
    print(f'\nコンピュータ内部で処理されている32ビットのデータ列:\n{combined_bin}')
except ValueError:
    print('正しいIPv4アドレスの形式（0〜255の数字を4つドットで区切る）で入力してください。')

## 4. サブネットマスク（ネットワーク部とホスト部）
IPアドレスは、**サブネットマスク** を使うことで「**ネットワーク部**」と「**ホスト部**」の2つに分けて管理されます。

- **同一セグメント（同じネットワーク内）：** ネットワーク部は **同じ値**、ホスト部は **重複しない値** にする。
- **異なるセグメント：** ネットワーク部を **違う値** にしなければならない。

例えば、`192.168.200.107/24` の `/24` は「先頭から24ビット目までがネットワーク部」を意味します（CIDR 表記）。

In [ ]:
# 【演習4】サブネットマスク計算ライブラリの体験
import ipaddress

cidr_input = input('IPアドレスとプレフィックスを入力してください (例: 192.168.128.10/24): ')

try:
    network = ipaddress.ip_network(cidr_input, strict=False)
    print(f'\n所属するネットワークの全体像（ホスト部がすべて0のアドレス）: {network.network_address}')
    print(f'適用されるサブネットマスク（十進数表現）: {network.netmask}')
    print(f'ブロードキャストアドレス: {network.broadcast_address}')
    print(f'このネットワークに接続できる端末数: {max(network.num_addresses - 2, 0)} 台（ネットワーク/ブロードキャストを除く）')
except ValueError:
    print("入力形式が正しくありません。'IPアドレス/ビット数' の形式で正確に入力してください。")

## 5. ルーターとデフォルトゲートウェイ
異なるネットワーク同士を相互接続し、パケットを中継する装置を **ルーター** と呼びます。ルーターは、宛先IPアドレスの「ネットワーク部」を確認し、次にどのルートへ配送すべきかを判断する **配達屋** のような役割を持ちます。

自分が所属するネットワーク（セグメント）の外側へ通信したいとき、パケットを最初に送り届けるネットワークの出口（ルーターの窓口）を **デフォルトゲートウェイ** と呼びます。

In [ ]:
# 【演習5】ルーターの「ネットワーク部」に基づくルーティング判断シミュレーション
my_network_part = '192.168.128'
print(f'あなたが所属するローカルネットワークは 【 {my_network_part}.0/24 】 です。')

dest_ip = input('データを送りたい相手のIPアドレスを入力してください (例: 192.168.128.11 または 192.168.144.10): ')

# ネットワーク部が一致するかどうかを簡易判定
if dest_ip.startswith(my_network_part + '.'):
    print('\n[ルーターの判断]: 宛先は【同じネットワーク内】にあります。')
    print('外部のルーターへパケットを送る必要はありません。L2スイッチ(ハブ)を介して直接相手に届けます。')
else:
    print('\n[ルーターの判断]: 宛先は【外側の異なるネットワーク】にあります。')
    print('パケットを一度『デフォルトゲートウェイ』へ転送し、外部ネットワークへと中継を依頼します。')

## 6. ドメイン名と DNS（Domain Name System）
コンピュータ同士の通信には IPアドレス（数字）が必要ですが、人間にとっては数字の羅列は覚えにくいので、`host-a` や `tu.chiba-u.ac.jp` のような **ホスト名（ドメイン名）** が使われます。

この「人間向けのドメイン名」と「コンピュータ向けのIPアドレス」の対応をネットワーク上で共有し、相互に変換（**名前解決**）する分散データベースが **DNS（Domain Name System）** です。ルート → トップレベル（jp, org, ...）→ 組織のDNSサーバ、と再帰的に問い合わせて答えに辿り着きます。

In [ ]:
# 【演習6】外部のDNSサーバに名前解決をリクエストしてみよう
import socket

print('--- DNS名前解決シミュレーター ---')
domain_name = input('IPアドレスを調べてみたいWebサイトのドメイン名を入力してください (例: chiba-u.ac.jp): ')

try:
    resolved_ip = socket.gethostbyname(domain_name)
    print('\nDNSサーバからの応答:')
    print(f'  入力されたドメイン名: {domain_name}')
    print(f'  解決されたIPアドレス: 【 {resolved_ip} 】')
    print('\nコンピュータはこの数字のアドレスを使って目的のサーバへ通信を開始します。')
except socket.gaierror:
    print('\n名前解決に失敗しました。ドメイン名が正しいか、またはネットワーク環境を確認してください。')

## 7. DHCP（Dynamic Host Configuration Protocol）
ネットワークに新しい端末を接続した際、通信に必要な設定（IPアドレス、サブネットマスク、デフォルトゲートウェイ、DNSサーバのアドレス など）を自動的に割り当てるプロトコルが **DHCP** です。

ケーブルを挿す／Wi-Fi に接続するだけで設定が完了するため、ネットワーク管理者の負担が大幅に減り、ユーザも面倒な設定なしで即座に通信を開始できます。

In [ ]:
# 【演習7】DHCPサーバによるIPアドレスの自動貸し出し（リース）疑似体験
import random

print('--- 端末をLANに接続します ---')
user_action = input('PCをネットワークに接続しますか？ (yes/no): ')

if user_action.strip().lower() in ['yes', 'y']:
    pool = [f'192.168.200.{i}' for i in range(100, 150)]
    allocated_ip = random.choice(pool)

    print('\n[DHCPサーバからのメッセージ]: 新しい端末を検知しました。以下の通信設定を自動適用します。')
    print(f'  貸出IPアドレス  : {allocated_ip}')
    print('  サブネットマスク: 255.255.255.0')
    print('  デフォルトGW   : 192.168.200.1')
    print('  DNSサーバ      : 192.168.200.2')
    print('  リース期間     : 例) 2025-05-27 → 2025-05-28（更新あり）')
    print('\n自動設定が成功しました！手動で数字を入力しなくてもインターネットに繋がります。')
else:
    print('接続がキャンセルされました。')

## 8. プライベートIPアドレスと NAT
IPアドレスには、利用される場所によって 2 種類があります。

- **プライベートIPアドレス：** 閉じた組織内で自由に使えるアドレス。以下の 3 つの帯が予約されています。
    - `10.0.0.0       〜 10.255.255.255   (10/8)`
    - `172.16.0.0     〜 172.31.255.255   (172.16/12)`
    - `192.168.0.0    〜 192.168.255.255  (192.168/16)`
- **グローバルIPアドレス：** インターネット上で世界中で重複しないように割り当てられる一意のアドレス。

プライベートIPのままではインターネットの世界と直接通信できないため、ルーターが境界でグローバルIPアドレスに書き換えて中継します。この技術を **NAT（Network Address Translation）** と呼びます。家庭用ルータには標準で NAT が組み込まれていることがほとんどです。

In [ ]:
# 【演習8】入力されたIPアドレスが「プライベート」か「グローバル」か判定しよう
import ipaddress

target_ip = input('判定したいIPアドレスを入力してください (例: 192.168.200.107 または 8.8.8.8): ')

try:
    ip_obj = ipaddress.ip_address(target_ip)

    if ip_obj.is_private:
        print(f'\n判定結果: 【 {target_ip} 】 は「プライベートIPアドレス」です。')
        print('これは社内や家庭内LANの中だけで有効なローカルな住所です。インターネットに出るにはNAT変換が必要です。')
    else:
        print(f'\n判定結果: 【 {target_ip} 】 は「グローバルIPアドレス」です。')
        print('インターネット上で世界中のルーターが直接ルートを判別できるパブリックな住所です。')
except ValueError:
    print('正しいIPアドレスの形式を入力してください。')

## 9. ポート番号によるアプリケーションの識別
パケットが目的のコンピュータ（IPアドレス）に到着した後、コンピュータ内部で動く「どのアプリケーション（処理）にデータを渡すか」を決定するために **ポート番号** という数字が使われます。

主要サービスごとに、世界共通で予約されているポート番号を **ウェルノウンポート（Well-known Ports）** と呼びます。

| ポート | プロトコル | 用途 |
|---|---|---|
| 20 / 21 | FTP | ファイル転送（データ／制御） |
| 22 | SSH | 安全な遠隔ログイン |
| 25 | SMTP | メール送信 |
| 53 | DNS (UDP/TCP) | 名前解決 |
| 67 / 68 | DHCP (UDP) | アドレス自動配布 |
| 80 | HTTP | Webサイトの閲覧 |
| 123 | NTP (UDP) | 時刻同期 |
| 443 | HTTPS | 暗号化Web通信 |

In [ ]:
# 【演習9】ウェルノウンポートのマッチングクイズ
print('--- ポート番号クイズ ---')
print('Webブラウザを開いてホームページを閲覧しようとするとき、通信相手のサーバの何番ポートに対してデータを送信すべきでしょうか？')
user_port = input('予約されているポート番号を半角数字で入力してください: ')

if user_port.strip() == '80':
    print('\n正解です！Web通信（HTTP）は標準で『80番ポート』が使われます。')
elif user_port.strip() in ['25', '22', '21', '53', '123', '443']:
    print(f'\n惜しい！{user_port}番もウェルノウンポートですが、別のプロトコル用です（表を再確認）。')
else:
    print('\n不正解です。講義スライドのポート番号一覧表（slide 42）を確認してみましょう。')

## 10. トランスポート層のプロトコル（TCP と UDP）
データを宛先のアプリケーションに送り届けるトランスポート層には、役割が異なる 2 つの主要プロトコルがあります。

- **TCP（Transmission Control Protocol）：** コネクション型。パケットが正しく届いたか確認し、欠落があれば再送、順序も並び替えるため **確実で信頼性が高い**。Web/メール/SSHなど。
- **UDP（User Datagram Protocol）：** コネクションレス型。確認・再送をせず一方的に送り続けるため **軽量で高速**。DNS／DHCP／NTP／音声・動画ストリーミングなど。

次のセルでは、ローカル（127.0.0.1）の中だけで完結する TCP ソケット通信を実演します。外部に通信は出ません。

In [ ]:
# 【演習10】Colabの内部で完結する、安全な疑似TCP通信テスト
# ※外部には通信しません。自分自身（127.0.0.1）の 9999 番ポートでデータを送受信します。
import socket

print('--- ローカルソケット通信シミュレーション ---')
message = input('通信テストとして送信するメッセージ文字列を入力してください: ')

# 1. 受け皿となるサーバ（待ち受け側）ソケットを作成
server_sim = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_sim.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)  # 再実行時のTIME_WAIT対策
server_sim.bind(('127.0.0.1', 9999))
server_sim.listen(1)

# 2. クライアント（送信側）ソケットを作成して接続し、メッセージを送信
client_sim = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_sim.connect(('127.0.0.1', 9999))
client_sim.sendall(message.encode('utf-8'))

# 3. サーバ側が接続を受け入れ、流れてきたデータを受信
connection, address = server_sim.accept()
received_data = connection.recv(1024).decode('utf-8')

print('\n--- 通信完了ログ ---')
print(f'接続元アドレス: {address}')
print(f'受信サーバ側が検知したメッセージ: 【 {received_data} 】')

# 通信終了後に安全にソケットをクローズ
client_sim.close()
connection.close()
server_sim.close()
print('安全にローカルソケット接続を終了・解放しました。')

## 11. まとめ＆発展：無線LANのセキュリティ
講義スライド 21〜22 では、無線 LAN（IEEE 802.11）のセキュリティについて以下が強調されました。

- **WPA2-PSK (AES) を使うのが最も安全。** 対応していない場合のみ WPA-PSK (AES)。
- **WEP は解読が非常に簡単。使ってはいけない。**
- 補助的な対策：MACアドレスフィルタリング／ESSID ステルス／any 接続拒否（これら単独では決定打にならない）。

### 発展クイズ
下のセルで、自宅ルータの暗号方式が安全かどうかをセルフチェックしてみましょう。

In [ ]:
# 【発展演習】無線LAN暗号方式セルフチェック
print('自宅のWi-Fiルータが使用している暗号方式を選んでください。')
print('  1) WPA3 / WPA3-SAE')
print('  2) WPA2-PSK (AES)')
print('  3) WPA-PSK (AES) または WPA/WPA2 mixed')
print('  4) WEP')
print('  5) 暗号化なし（オープン）')
choice = input('番号を入力: ').strip()

verdict = {
    '1': '◎ 最新世代。十分に安全です。',
    '2': '○ 講義での推奨。安全に運用できます。',
    '3': '△ 妥協案。可能なら WPA2-PSK(AES) 以上に切替を。',
    '4': '× 危険。解読が容易です。今すぐ WPA2 以上へ変更してください。',
    '5': '× 通信が丸見え。公共用途以外では絶対に使わないこと。',
}
print('\n判定:', verdict.get(choice, '入力が不正です。1〜5 で答えてください。'))

---
## 参考文献
- 竹下 隆史・村山 公保・荒井 透・苅田 幸雄 (2012). 『マスタリングTCP/IP 入門編 第5版』オーム社.
- 竹下 隆史・村山 公保・荒井 透・苅田 幸雄 (2019). 『マスタリングTCP/IP 入門編 第6版』オーム社.
- KEYENCE. Wi-Fi 規格と最新動向. https://www.keyence.co.jp/ss/products/controls/wifi/basis/latest_standards.jsp
- 平成25年春期 ITパスポート試験公開問題 問68.
